# 🎬 Video Creator — AI Image Server

This notebook runs a high-quality AI image generation server on Google's free GPU and exposes it via ngrok so your Next.js app can call it.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom
3. Copy the ngrok URL from Cell 4 output
4. Paste it into your app's `.env.local` as `COLAB_IMAGE_URL=https://xxxx.ngrok-free.app`
5. Keep this tab open while generating videos

**Model:** SDXL Base 1.0 — cinematic quality, ~4-5s per image on T4 GPU

In [ ]:
# Cell 1 — Install dependencies (~2 minutes first time)
!pip install -q diffusers transformers accelerate xformers flask flask-cors pyngrok omegaconf

In [ ]:
# Cell 2 — Load SDXL model (~5 minutes to download first time, instant after)
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

print('Loading SDXL Base 1.0...')

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant='fp16',
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()

try:
    pipe.enable_xformers_memory_efficient_attention()
    print('xformers enabled')
except Exception:
    pass

# Warmup — first inference is slow, pre-run to avoid timeout on first real request
print('Warming up...')
with torch.no_grad():
    pipe('warmup image', width=512, height=512, num_inference_steps=1)

print('\n✅ Model ready!')

In [ ]:
# Cell 3 — Flask API server
import io
import time
import torch
from flask import Flask, request, Response, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

# SDXL recommended portrait/landscape sizes (must be multiples of 64)
PORTRAIT_W, PORTRAIT_H   = 768, 1344   # 9:16 Shorts
LANDSCAPE_W, LANDSCAPE_H = 1344, 768   # 16:9 Video

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'sdxl-base-1.0', 'device': 'cuda'})

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.get_json(force=True)
        prompt = data.get('prompt', 'cinematic scene, dramatic lighting')
        width  = int(data.get('width',  1080))
        height = int(data.get('height', 1920))
        steps  = int(data.get('steps',  25))

        # Map to SDXL-friendly sizes
        if height > width:
            w, h = PORTRAIT_W, PORTRAIT_H
        else:
            w, h = LANDSCAPE_W, LANDSCAPE_H

        # Append quality boosters
        full_prompt = f'{prompt}, cinematic photography, professional, sharp focus, high resolution'
        neg_prompt  = 'blurry, low quality, distorted, ugly, watermark, text, logo, duplicate'

        t0 = time.time()
        with torch.no_grad():
            result = pipe(
                prompt=full_prompt,
                negative_prompt=neg_prompt,
                width=w,
                height=h,
                num_inference_steps=steps,
                guidance_scale=7.5,
            )
        elapsed = round(time.time() - t0, 1)
        print(f'Generated in {elapsed}s: {prompt[:60]}')

        img = result.images[0]
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=92)
        buf.seek(0)

        return Response(
            buf.read(),
            mimetype='image/jpeg',
            headers={'X-Generation-Time': str(elapsed)}
        )

    except Exception as e:
        print(f'Error: {e}')
        return jsonify({'error': str(e)}), 500

print('Flask API defined')

In [ ]:
# Cell 4 — Start server + ngrok tunnel
# Paste your ngrok authtoken here (ngrok.com → free account → Your Authtoken)
NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'

from pyngrok import ngrok, conf
import threading

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()  # kill any existing tunnels

# Start Flask in background thread
thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)
)
thread.daemon = True
thread.start()

# Open ngrok tunnel
tunnel = ngrok.connect(5000, 'http')
url = tunnel.public_url

print('=' * 60)
print('🚀 AI Image Server is LIVE!')
print('=' * 60)
print(f'\n📡 Colab URL: {url}')
print(f'\nAdd this to your .env.local file:')
print(f'\nCOLAB_IMAGE_URL={url}')
print('\n⚠️  Keep this tab open while generating videos!')
print('\nTest it: ' + url + '/health')

In [ ]:
# Optional Cell 5 — Test generation (run to verify it works)
import requests

response = requests.post(f'{url}/generate', json={
    'prompt': 'extreme close-up of a young Indian man counting rupee notes, dramatic shadow lighting, cinematic',
    'width': 1080,
    'height': 1920,
    'steps': 20
})

print(f'Status: {response.status_code}')
print(f'Content-Type: {response.headers.get("Content-Type")}')
print(f'Generation time: {response.headers.get("X-Generation-Time")}s')

# Display the image
from IPython.display import Image as IPImage, display
display(IPImage(data=response.content, width=300))